<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w5_reranking/llm_260407_reranking_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Day 1 - Reranking 기법 입문
**2026-04-07 (화) | 모두의연구소 LLM 서비스 과정 6기**

---

## 오늘 배울 내용
| 순서 | 주제 | 핵심 |
|------|------|------|
| 1 | Reranking 개념 | 왜 1차 검색만으로는 부족한가? |
| 2 | Keyword Rerank | 키워드 가중치로 순위 재조정 |
| 3 | BM25 Reranker | TF-IDF 기반 정밀 재랭킹 |
| 4 | LLM Rerank (Pointwise) | LLM에게 문서별 관련성 점수 매기기 |
| 5 | Hybrid Rerank | BM25 + LLM 결합 |
| 6 | LLM Listwise Rerank | LLM에게 전체 문서 한번에 정렬시키기 |
| 7 | Score Filter | threshold / dynamic 필터링 |

---

### w4 복습 (지난주 핵심)
- RAG 평가 메트릭: Precision@K, Recall@K, MRR, NDCG
- Hybrid Search: 벡터 + 키워드 검색 결합
- Query Expansion: 쿼리를 확장해서 검색 품질 향상

### 이번 주 핵심 질문
> 1차 검색으로 후보 문서를 뽑은 뒤, **어떻게 순서를 다시 매길 것인가?**

## 0. 환경 설정 및 Import

In [ ]:
# -- Colab 사용 시 아래 주석 해제 --
# !pip install openai langchain-openai langchain-community faiss-cpu python-dotenv
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 실습용 문서 준비

8개의 NLP/LLM 관련 문서를 Document 객체로 만들고, FAISS 벡터스토어와 BM25 리트리버를 생성합니다.

In [ ]:
# 8개의 NLP/LLM 관련 실습용 문서
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [ ]:
# FAISS 벡터스토어 생성 (임베딩 기반 검색용)
vectorstore = FAISS.from_documents(documents, embeddings_model)

# BM25 리트리버 생성 (키워드 기반 검색용, 상위 5개 반환)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [ ]:
# 각 문서의 임베딩 벡터를 미리 계산해서 딕셔너리에 저장
# (나중에 유사도 비교 등에 활용 가능)
doc_embeddings = {}

def get_embedding(text):
    """텍스트를 임베딩 벡터(numpy array)로 변환"""
    return np.array(embeddings_model.embed_query(text))

for doc in documents:  # 주의: 원래 코드는 doc_embeddings를 순회 (빈 dict) -> documents로 수정
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

## 2. Reranking이 필요한 이유

### Bi-encoder vs Cross-encoder

**비유: 서류 심사 vs 면접**
- **Bi-encoder (1차 검색)**: 이력서(쿼리)와 회사 요구사항(문서)을 각각 점수화한 뒤 비교하는 것. 빠르지만, 둘 사이의 "케미"를 놓칠 수 있음.
- **Cross-encoder (Reranking)**: 면접에서 지원자와 면접관이 직접 대화하며 궁합을 보는 것. 느리지만 정확.

### 왜 필요한가?
임베딩 기반 검색(Bi-encoder)은 쿼리와 문서를 **따로따로** 벡터로 만든 뒤 유사도를 비교합니다.
- 장점: 빠름 (문서 벡터 미리 계산 가능)
- 단점: 쿼리-문서 간 **상호작용**을 볼 수 없음

예) "트랜스포머와 BERT의 관계"라는 쿼리 -> 벡터 검색은 "트랜스포머"와 가까운 문서를 먼저 가져옴. 하지만 실제로 가장 관련 있는 건 BERT가 트랜스포머의 인코더 부분을 사용한다는 d2 문서.

### Reranking 아키텍처
```
검색(1차) -> 후보문서 5개 -> Rerank(Cross-encoder/LLM) -> 재정렬된 문서 -> Generator
```
1차 검색은 **싸고 빠르게** 후보를 추림 -> Reranking은 **비싸지만 정확하게** 순서를 다시 매김

In [ ]:
# -- 수업 중 필기 메모 (Bi-encoder vs Cross-encoder 개념) --

# [Bi-encoder: 일반 벡터 검색]
# query ---(emb_model)---> query_emb
# doc1  ---(emb_model)---> doc1_emb
# -> 각각 따로 임베딩 후 유사도 비교

# [Cross-encoder: 쿼리+문서를 함께 인코딩]
# (query + doc1) ---(emb_model)---> (query_doc1_emb)
# (query + doc2) ---(emb_model)---> (query_doc2_emb)
# -> 쿼리와 문서의 상호작용까지 반영

# [LLM 기반 Reranking]
# query, doc1 --(LLM)--> score 또는 true/false
# -> 프롬프트로 관련성 판단을 맡김

In [ ]:
# -- Reranking 파이프라인 전체 흐름 --
# 검색 -> doc1, doc2, ... doc5 -> 후보문서 -> re-rank ----(Cross-encoder / LLM)--> re-ranked -> generate

## 3. 벡터 검색 (1차 검색)

FAISS의 `similarity_search_with_score`는 **거리(distance)** 기반으로 스코어를 반환합니다.
- 거리가 가까울수록 유사 -> 값이 작을수록 좋음
- 이를 **유사도** 기반(값이 클수록 좋음)으로 변환: `1 / (1 + distance)`
  - distance=0 -> 유사도=1.0 (완벽 일치)
  - distance가 클수록 -> 유사도가 0에 가까워짐

In [ ]:
def vector_search(query, vectorstore, top_k=5):
    """벡터 검색 후 거리 -> 유사도로 변환하여 반환"""
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    # score는 거리(distance)이므로, 역수를 취해 유사도로 변환
    # +1은 분모가 0이 되는 것을 방지
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

### 1차 검색 결과 관찰
- d1(트랜스포머)이 1위, d2(BERT)가 2위로 나옴
- 하지만 "관계"를 물었으므로 d2(BERT는 트랜스포머 인코더 기반)가 더 관련성 높을 수 있음
- 이것이 Reranking이 필요한 이유!

## 4. Keyword Rerank (키워드 기반 재랭킹)

**가장 단순한 Reranking**: 1차 검색 결과에서, 쿼리 키워드가 문서에 많이 등장하면 가점을 줌.

**비유: 시험 채점에 가산점**
- 1차 시험(벡터 검색) 점수가 있고, 특정 키워드(자격증)가 있으면 추가 점수를 주는 것.
- 기본 점수 + 0.1 * 키워드_일치_수

In [ ]:
def keyword_rerank(query, search_results):
    """키워드 매칭 기반으로 검색 결과를 재랭킹
    - 쿼리 키워드가 문서에 많이 등장할수록 가점 부여
    - new_score = original_score + 0.1 * 키워드_일치_수
    """
    # 쿼리를 소문자로 바꾸고 공백으로 분리 -> 토큰 집합
    query_terms = set(query.lower().split())
    reranked = []

    for doc, orig_score in search_results:
        # 문서도 동일하게 토큰화
        doc_terms = doc.page_content.lower().split()
        # 문서 토큰 중 쿼리 키워드와 겹치는 개수
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)
        # 기존 점수에 키워드 매칭 가점 추가 (0.1 * 겹치는 수)
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))

    # 새로운 점수 기준으로 내림차순 정렬
    reranked.sort(key = lambda x: x[1], reverse=True)
    return reranked

In [ ]:
reranked = keyword_rerank(query, results)
reranked

# 참고: 한국어는 조사가 붙어있어서 ("트랜스포머는" vs "트랜스포머와")
# 단순 split으로는 키워드 매칭이 잘 안 됨
# -> 실무에서는 형태소 분석기(konlpy 등) 사용 권장

## 5. 순위 변화 추적 함수

Reranking 전후 순위가 어떻게 바뀌었는지 DataFrame으로 확인합니다.

In [ ]:
def rank_change(original_results, reranked_results):
    """원본 검색 결과와 리랭킹 결과의 순위 변화를 DataFrame으로 반환
    - before: 원래 순위
    - after: 리랭킹 후 순위
    - change: 순위 변화 (양수면 상승, 음수면 하락)
    """
    # 원본 순위: (doc, score) 튜플에서 doc_id -> 순위 매핑
    orig_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _) in enumerate(original_results)}
    # 리랭킹 순위: (doc, new_score, orig_score) 튜플에서 doc_id -> 순위 매핑
    new_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _, _) in enumerate(reranked_results)}

    changes = []
    for doc_id in orig_ranks:
        old_r = orig_ranks[doc_id]
        new_r = new_ranks.get(doc_id, -1)  # -1: 리랭킹 결과에 없는 경우 (top_k가 다를 때)

        changes.append({
            'doc_id' : doc_id,
            'before' : old_r,
            'after' : new_r,
            'change' : old_r - new_r  # 양수면 순위 상승
        })

    return pd.DataFrame(changes).sort_values('after')

In [ ]:
df = rank_change(results, reranked)
df

# 결과: change가 모두 0 -> 키워드 매칭이 안 돼서 순위 변화 없음
# (한국어 조사 문제: "트랜스포머와" != "트랜스포머는")

## 6. BM25 Reranker (TF-IDF 기반 재랭킹)

### TF-IDF 복습
- **TF (Term Frequency)**: 단어가 문서에 많이 나올수록 중요 -> 값이 커짐
- **IDF (Inverse Document Frequency)**: 모든 문서에 흔하게 나오는 단어(은/는/이/가)는 덜 중요 -> 역수를 취해 가치 하락
- **TF-IDF = TF x IDF**: 특정 문서에서 자주 나오면서, 전체적으로는 드문 단어일수록 높은 점수

### BM25
TF-IDF에 **두 가지 파라미터**를 추가해서 보정한 알고리즘:
- `k1` (기본 1.5): TF의 포화도 조절. 클수록 TF를 더 중요시
- `b` (기본 0.75): 문서 길이 정규화. 1에 가까울수록 긴 문서에 불리

**비유: 맛집 평가**
- TF = 이 식당 리뷰에 "맛있다"가 몇 번 나오나? (많을수록 좋음)
- IDF = "맛있다"가 모든 식당 리뷰에 다 나오면 별 의미 없음 (흔한 표현은 가치 하락)
- BM25 = 리뷰 길이까지 고려 (100자 리뷰에서 3번 vs 10자 리뷰에서 3번은 다름)

In [ ]:
class BM25Reranker:
    """BM25 알고리즘 기반 재랭킹
    - 1차 검색 결과를 키워드(TF-IDF) 관점에서 다시 점수 매김
    - k1: TF 포화도 파라미터 (기본 1.5)
    - b: 문서 길이 정규화 파라미터 (기본 0.75)
    """
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        # 1차 검색 결과에서 문서만 추출
        docs = [doc for doc, _ in search_results]
        # 각 문서를 토큰화 (소문자 + 공백 분리)
        tokenized = [doc.page_content.lower().split() for doc in docs]

        # 전체 문서의 평균 길이 (BM25의 b 파라미터에서 사용)
        avg_dl = np.mean([len(t) for t in tokenized])

        # DF (Document Frequency): 각 토큰이 몇 개 문서에 등장하는지
        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):  # set으로 중복 제거: 한 문서에서 여러번 나와도 DF는 1
                df_count[t] += 1
        N = len(docs)  # 전체 문서 수

        # 쿼리도 동일하게 토큰화
        query_tokens = query.lower().split()
        scored = []

        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])  # 현재 문서의 길이
            tf_count = Counter(tokenized[i])  # 현재 문서의 TF (단어별 등장 횟수)

            for qt in query_tokens:
                tf = tf_count.get(qt, 0)  # 쿼리 토큰의 TF
                if tf == 0:
                    continue  # 이 단어가 문서에 없으면 스킵

                df = df_count.get(qt, 0)  # 이 단어의 DF
                # IDF 계산: log((N - df + 0.5) / (df + 0.5) + 1)
                # 흔한 단어(df 큼) -> IDF 작음, 희귀한 단어(df 작음) -> IDF 큼
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
                # BM25 보정: k1과 b로 TF와 문서 길이를 조절
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / avg_dl)
                score += idf * numerator / denominator

            scored.append((doc, score))

        # 점수 내림차순 정렬
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

In [ ]:
# BM25 Reranker 테스트
bm25_reranker = BM25Reranker()
bm25_results = bm25_reranker.rerank(query, results)

In [ ]:
bm25_results

# 결과: 모든 점수가 0.0 -> 쿼리 토큰("트랜스포머와", "bert의", "관계")이
# 문서에 정확히 일치하는 형태로 없기 때문
# -> 한국어에서는 형태소 분석 없이 단순 split은 한계가 있음

## 7. LLM Rerank (Pointwise)

LLM에게 **각 문서의 관련성 점수(0.0~1.0)**를 매기게 합니다.

**비유: 전문가 심사위원**
- BM25나 키워드는 기계적으로 단어 매칭만 함 (규칙 기반)
- LLM은 "트랜스포머와 BERT의 관계"의 **의미**를 이해하고, BERT가 트랜스포머의 인코더 부분을 사용한다는 것을 알고 있음
- 따라서 d2(BERT)에 가장 높은 관련성 점수를 줄 수 있음

### Pointwise vs Listwise
- **Pointwise**: 문서를 1개씩 LLM에 넣어서 점수 매김 -> 정확하지만 문서간 비교 불가
- **Listwise**: 여러 문서를 한번에 넣고 정렬 요청 -> 비교 가능하지만 할루시네이션 위험

여기서는 먼저 **모든 문서를 한번에 넣고** JSON으로 점수를 받는 방식을 구현합니다.

In [ ]:
def llm_rerank(query, search_results, top_k=3):
    """LLM을 이용한 Pointwise 재랭킹
    - 모든 문서를 프롬프트에 넣고, 각 문서별 관련성 점수(0.0~1.0)를 JSON으로 받음
    - top_k개만 반환
    """
    # 문서들을 [id] 내용 형태로 텍스트 변환
    docs_text = '\n'.join(
        f"[{doc.metadata['id']}] {doc.page_content}" for doc, _ in search_results
    )

    # JSON 응답 형식 가이드: {"d1": 0.0-1.0, "d2": 0.0-1.0, ...}
    score_template = ", ".join(f'"{ doc.metadata["id"] }": 0.0-1.0' for doc, _ in search_results)

    # LCEL 체인 구성: 프롬프트 -> LLM -> 문자열 파싱
    scoring_chain = ChatPromptTemplate.from_messages([
        ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
        ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0~1.0으로 평가하세요.

        쿼리 : {query}

        문서들 :
        {docs_text}

        "스트링으로 묶지말고" JSON으로 답하세요:
        {{"scores" : {{{score_template}}}}}""")
    ]) | llm | StrOutputParser()

    # 체인 실행
    result = scoring_chain.invoke({
        'query' : query,
        'docs_text' : docs_text,
        'score_template' : score_template
    })

    # LLM 응답에서 ```json ... ``` 마크다운 제거
    cleaned = result.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned.replace('```json', '')
        cleaned = cleaned.replace('```', '')

    # JSON 파싱 -> scores 딕셔너리 추출
    parsed = json.loads(cleaned)
    scores = parsed.get('scores', {})

    # 원본 점수와 함께 새 점수 리스트 생성
    scored = []
    for doc, orig in search_results:
        rerank_score = scores.get(doc.metadata['id'], 0.0)
        scored.append((doc, float(rerank_score), orig))  # (문서, 새점수, 원래점수)

    # 새 점수 기준 내림차순 정렬, 상위 top_k개 반환
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

In [ ]:
query  # 확인: '트랜스포머와 BERT의 관계'

In [ ]:
results  # 1차 벡터 검색 결과 확인

In [ ]:
# LLM 리랭킹 실행 (API 호출 발생 -> 시간/비용 소모)
llm_results = llm_rerank(query, results, top_k=3)

In [ ]:
llm_results

# 기대 결과: d2(BERT)가 1위로 올라옴!
# LLM은 "BERT는 트랜스포머 인코더 기반"이라는 의미를 이해하므로
# "트랜스포머와 BERT의 관계"에 가장 관련 높다고 판단

## 8. Hybrid Rerank (BM25 + LLM 결합)

**비유: 시험 최종 점수 = 필기시험(BM25) * 30% + 면접(LLM) * 70%**

두 가지 리랭킹 점수를 가중 합산합니다. 이때 **정규화(Normalize)**가 핵심!

### 왜 정규화가 필요한가?
- BM25 점수 범위: 0~100 (가정)
- LLM 점수 범위: 0~1
- 그냥 더하면 BM25가 무조건 이김 -> 하이브리드 의미 없음
- Min-Max 정규화로 두 점수를 모두 0~1 범위로 맞춘 뒤 가중 합산

In [ ]:
def hybrid_rerank(query, search_results, bm25_weight=0.3):
    """BM25 + LLM 하이브리드 재랭킹
    - bm25_weight: BM25 점수의 가중치 (나머지는 LLM)
    - 두 점수를 Min-Max 정규화 후 가중 합산
    """
    # BM25 리랭킹 실행
    bm25 = BM25Reranker()
    bm25_scored = bm25.rerank(query, search_results)
    bm25_map = {doc.metadata['id']: score for doc, score in bm25_scored}

    # LLM 리랭킹 실행 (전체 문서 대상)
    llm_scored = llm_rerank(query, search_results, top_k=len(search_results))
    llm_map = {doc.metadata['id']: score for doc, score, _ in llm_scored}

    # Min-Max 정규화 함수 (0~1 범위로 스케일 맞추기)
    def normalize(scores_dict):
        vals = list(scores_dict.values())
        min_, max_ = min(vals), max(vals)
        range_ = max_ - min_ if max_ > min_ else 1e-8  # 0으로 나누기 방지
        return {k: (v - min_) / range_ for k, v in scores_dict.items()}

    bm25_norm = normalize(bm25_map)
    llm_norm = normalize(llm_map)

    # 가중 합산: score = bm25_weight * bm25_norm + (1-bm25_weight) * llm_norm
    combined = []
    for doc, _ in search_results:
        did = doc.metadata['id']
        score = bm25_weight * bm25_norm.get(did, 0) + (1 - bm25_weight) * llm_norm.get(did, 0)
        combined.append((doc, score))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined

In [ ]:
# 하이브리드 리랭킹 실행
hybrid = hybrid_rerank(query, results)

In [ ]:
hybrid

# 실무 팁: bm25_weight를 Grid Search로 최적값 탐색
# 예) 0.1, 0.2, 0.3, ... 0.9로 실험해서 가장 좋은 성능의 weight 선택

## 9. Pointwise vs Listwise 리랭킹

| | Pointwise (1개씩) | Listwise (N개 한번에) |
|---|---|---|
| **장점** | 정확, 이유 명확, 병렬 처리 가능 | 비용 저렴, 문서간 비교 가능 |
| **단점** | 비용 높음, 문서간 비교 불가 | 할루시네이션 위험, 컨텍스트 길이 제한 |
| **추천 상황** | 빠른 응답 필요시 (병렬) | 5개 이하 문서, 비용 절약 |

**비유**
- Pointwise: 면접관이 지원자를 1명씩 면접 -> 개별 평가는 정확하지만 비교가 어려움
- Listwise: 면접관이 지원자 5명을 한 방에 모아서 그룹 면접 -> 비교는 쉽지만 개별 평가가 부정확할 수 있음

In [ ]:
# -- 수업 중 필기 메모 --
# pointwise reranking (1개씩 넣는다)
# - 정확, 이유를 명확하게 설명, 빠르다(병렬)
# - 비용, 문서간 비교를 못한다

# listwise reranking  (N개씩 넣는다 : 이 문서들을 정렬해줘, 스코어링해줘)
# - 비용 저렴, 문서간 비교를 한다
# - 할루시네이션/불명확한 설명, 컨텍스트(많은 문서)

# 가이드라인:
# 5개 이하 -> listwise가 효율적
# 빠른 응답이 필요한 경우 -> pointwise로 병렬 호출

## 10. LLM Listwise Rerank

LLM에게 문서 전체를 보여주고 **관련성 순서대로 정렬**하라고 요청합니다.
- 점수 대신 **순서(번호)**를 받아서 랭킹으로 변환
- 예: "3,1,5,2,4" -> 3번 문서가 1위, 1번 문서가 2위, ...

In [ ]:
def llm_listwise_rerank(query, search_results):
    """LLM Listwise 재랭킹
    - 모든 문서를 한번에 보여주고 관련성 순서대로 번호를 받음
    - 순위를 점수로 변환: 1위=1.0, 2위=0.9, 3위=0.8, ...
    """
    # 문서를 번호와 함께 텍스트로 변환 (내용은 80자까지만)
    docs_text = '\n'.join(
        f"{i+1}. [{doc.metadata['id']}] {doc.page_content[:80]}"
        for i, (doc, _) in enumerate(search_results)
    )

    # 체인: 관련성 순서대로 번호 나열 요청
    ranking_chain = ChatPromptTemplate.from_messages([
        ('system', "당신은 문서 랭킹 시스템입니다"),
        ('human', """다음 문서들을 쿼리와의 관련성 순서로 정렬하세요.

        쿼리 : {query}

        문서들 : {docs_text}

        가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3,1,5,2,4):""")
    ]) | llm | StrOutputParser()

    result = ranking_chain.invoke({'query': query, 'docs_text': docs_text})

    # "2,1,3,5,4" 같은 문자열을 정수 리스트로 변환
    order = [int(x.strip()) for x in result.strip().split(',')]

    # 순위 -> 점수 변환: 1위=1.0, 2위=0.9, ...
    docs_list = [doc for doc, _ in search_results]
    reranked = []
    for rank, idx in enumerate(order):
        if 1 <= idx <= len(docs_list):  # 유효 범위 체크
            docs = docs_list[idx - 1]
            reranked.append((docs, 1.0 - rank * 0.1))

    return reranked

In [ ]:
# Listwise 리랭킹 실행
listwise = llm_listwise_rerank(query, results)

In [ ]:
listwise

# 기대: d2(BERT)가 1위, d1(트랜스포머)이 2위
# LLM이 "관계"라는 맥락을 이해하고 BERT-트랜스포머 관계를 파악

## 11. Score Filter (점수 기반 필터링)

리랭킹으로 점수를 매긴 뒤, **어떤 문서를 Generator에 넘길지** 결정하는 전략입니다.

### 4가지 필터링 전략
| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| Fixed Threshold | 고정 점수 이상만 통과 (예: 0.5 이상) | 단순, 예측 가능 | 쿼리마다 분포가 달라 비효율적 |
| Dynamic Threshold | 평균 - n*표준편차 이상만 통과 | 분포에 적응 | 계산 추가 필요 |
| Score Gap | 점수가 크게 떨어지는 지점에서 자름 | 자연스러운 구분 | 구현 복잡 |
| Top-K | 항상 상위 K개만 사용 | 가장 단순 | 관련 없는 문서도 포함 가능 |

**비유: 대학 합격선**
- Fixed Threshold: "80점 이상만 합격" -> 어떤 해에는 합격자 0명, 어떤 해에는 전원 합격
- Dynamic Threshold: "상위 2 표준편차 이내만 합격" -> 매년 적절한 인원 선발
- Score Gap: "점수가 갑자기 10점 이상 떨어지는 지점에서 커트" -> 자연스러운 그룹 분리

In [ ]:
class ScoreFilter:
    """리랭킹 점수 기반 문서 필터링 클래스
    - fixed_threshold: 고정 점수 기준 필터링
    - dynamic_threshold: 평균/표준편차 기반 동적 필터링
    """

    def fixed_threshold(scored_docs, threshold=0.5):
        """고정 threshold 이상인 문서만 반환
        예) threshold=0.5 -> 점수 0.5 이상만 통과
        """
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

    def dynamic_threshold(scored_docs, std_factor):
        """동적 threshold 필터링
        threshold = 평균 - std_factor * 표준편차
        - std_factor=1: 약 84%의 문서 통과 (정규분포 가정)
        - std_factor=2: 약 97.5%의 문서 통과
        - std_factor 작을수록 더 엄격한 필터링
        """
        scores = [s for _, s in scored_docs]
        threshold = np.mean(scores) - std_factor * np.std(scores)
        return [(doc, s) for doc, s in scored_docs if s >= threshold]

In [ ]:
# -- Score Filter 테스트 예시 --
# listwise 결과로 필터링 테스트

# Fixed: 0.8 이상만
print("=== Fixed Threshold (0.8) ===")
filtered_fixed = ScoreFilter.fixed_threshold(listwise, threshold=0.8)
for doc, s in filtered_fixed:
    print(f"  [{doc.metadata['id']}] score={s:.2f} | {doc.page_content[:50]}")

print()

# Dynamic: 평균 - 1*표준편차 이상만
print("=== Dynamic Threshold (std_factor=1) ===")
filtered_dyn = ScoreFilter.dynamic_threshold(listwise, std_factor=1)
for doc, s in filtered_dyn:
    print(f"  [{doc.metadata['id']}] score={s:.2f} | {doc.page_content[:50]}")

---

## 핵심 정리

### 오늘 배운 Reranking 기법 전체 흐름
```
사용자 쿼리
  -> 1차 검색 (벡터/BM25, 빠르고 저렴)
  -> 후보 문서 5~10개
  -> Reranking (Cross-encoder / LLM, 느리지만 정확)
  -> Score Filtering (threshold / top-k)
  -> Generator (LLM 답변 생성)
```

### 리랭킹 방법 비교
| 방법 | 속도 | 정확도 | 비용 | 언제 쓰나? |
|------|------|--------|------|------------|
| Keyword Rerank | 빠름 | 낮음 | 무료 | 빠른 프로토타입 |
| BM25 Reranker | 빠름 | 중간 | 무료 | 키워드 매칭 중요할 때 |
| LLM Pointwise | 느림 | 높음 | 비쌈 | 정확한 개별 평가 필요 |
| LLM Listwise | 중간 | 높음 | 중간 | 문서간 상대 비교 필요 |
| Hybrid | 느림 | 매우 높음 | 비쌈 | 최고 성능이 필요할 때 |

### 다음 시간 예고
- Score Gap 기반 필터링 구현
- ContextualCompressionRetriever (LangChain 내장 리랭킹)
- RAG 전체 파이프라인에 Reranking 적용